In [1]:
# -*- coding: utf-8 -*-
import sys
import os
import glob
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import gaussian_kde 

# ==============================================================================
# PERTAHANAN MURNI CPU & ANTI-LEAK UNTUK MACBOOK M3 PRO
# ==============================================================================
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Mematikan seluruh warning log internal
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU') 
from tensorflow import keras
from tensorflow.keras import backend as K
# ==============================================================================

# --- KONFIGURASI PATH UTAMA ---
BASE_REP = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo'
if BASE_REP not in sys.path:
    sys.path.append(BASE_REP)

from Library import utils, dataset

# Direktori Sumber Data Master 3C Indonesia
DIR_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_gempa_9s"
DIR_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_noise_9s"

def predict_and_evaluate_1c_batch(model, batch_waves, batch_labels, kde_noise, kde_le):
    """
    Isolasi fungsi kloter 1C untuk memaksa penghapusan referensi variabel lokal
    dari memori RAM setelah fungsi selesai dieksekusi.
    """
    num_points = 700
    batch_input = np.array(batch_waves, dtype=np.float32).reshape(-1, num_points, 1)
    
    with tf.device('/CPU:0'):
        embeddings = model.predict_on_batch(batch_input)
    
    # Transposisi murni ruang laten 1D untuk pencocokan kurva PDF 1D
    emb_T = embeddings.T 
    like_noise = kde_noise.pdf(emb_T)
    like_le = kde_le.pdf(emb_T)
    
    # KEPUTUSAN BINER MURNI: Evaluasi murni via argmax asli Zhi Geng (0 = Noise, 1 = LE)
    likelihoods = np.vstack([like_noise, like_le])
    preds = np.argmax(likelihoods, axis=0)
    
    local_tp, local_tn, local_fp, local_fn = 0, 0, 0, 0
    for t_lbl, p_lbl in zip(batch_labels, preds):
        if t_lbl == 1 and p_lbl == 1: local_tp += 1
        elif t_lbl == 0 and p_lbl == 0: local_tn += 1
        elif t_lbl == 0 and p_lbl == 1: local_fp += 1
        elif t_lbl == 1 and p_lbl == 0: local_fn += 1
        
    return local_tp, local_tn, local_fp, local_fn

if __name__ == "__main__":
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20") 
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    # --------------------------------------------------------------------------
    # FASE 0: LOAD MODEL & BANGUN KDE BINER MURNI (Z-AXIS ONLY)
    # --------------------------------------------------------------------------
    print("[INFO] Memuat Model dan Membangun Kurva PDF 1D Murni (Z-Axis)...")
    with tf.device('/CPU:0'):
        embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    
    available_keys = list(embedding_Z.keys())
    k_noise = next((k for k in available_keys if k.lower() in ['noise', 'no']), available_keys[0])
    k_le = next((k for k in available_keys if k.lower() in ['le', 'earthquake', 'eq']), available_keys[1] if len(available_keys)>1 else available_keys[0])

    # Membangun density fungsi murni biner NO vs LE
    kde_noise = gaussian_kde(np.array(embedding_Z[k_noise]).T)
    kde_le = gaussian_kde(np.array(embedding_Z[k_le]).T)
    
    del embedding_Z
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 1: PERSIAPAN DATASET UJI INDONESIA
    # --------------------------------------------------------------------------
    files_gempa = glob.glob(os.path.join(DIR_GEMPA, "*.npy"))
    files_noise = glob.glob(os.path.join(DIR_NOISE, "*.npy"))
    
    print(f"[INFO] Total Data Uji Indonesia Terdeteksi: {len(files_gempa):,} Gempa | {len(files_noise):,} Noise")
    
    all_files = [(f, 1) for f in files_gempa] + [(f, 0) for f in files_noise]
    np.random.seed(42)
    np.random.shuffle(all_files)
    
    # SHIELD MEMORY ANTI-LEAK 1: Konversi ke list datar terpisah untuk memutus referensi objek tuple masif
    file_paths = [item[0] for item in all_files]
    file_labels = [item[1] for item in all_files]
    
    del all_files, files_gempa, files_noise
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: INFERENSI TERMINAL AMAN MEMORI MURNI 1C ZHI GENG
    # --------------------------------------------------------------------------
    num_points = 700  # Jendela kaku 7 detik @ 100 Hz
    BUFFER_SIZE = 128
    buffer_waves = []
    buffer_labels = []

    TP, TN, FP, FN = 0, 0, 0, 0
    FILE_KORUP = 0

    print(f"\n[INFO] Memulai eksekusi 1C murni Indonesia untuk {len(file_paths):,} sampel...")
    
    for idx in tqdm(range(len(file_paths)), desc="Indonesia Pure 1C 100K"):
        try:
            wave_3c = np.load(file_paths[idx])
            
            # Validasi awal bentuk array data master 3C Indonesia
            if not np.isfinite(wave_3c).all() or wave_3c.shape != (700, 3):
                FILE_KORUP += 1
                continue

            # MURNI ZHI GENG: Slicing murni komponen vertikal Z-axis (indeks 2)
            z_component = wave_3c[:, 2]
            true_label = file_labels[idx]
            
            # PRE-PROCESSING MURNI: Detrending & Normalisasi Maksimum (Tanpa Bandpass Filter)
            z_component = z_component - np.mean(z_component)
            norm_val = np.max(np.abs(z_component))
            if norm_val > 0: 
                z_component /= norm_val
            
            buffer_waves.append(z_component)
            buffer_labels.append(true_label)
            
            if len(buffer_waves) == BUFFER_SIZE:
                # Tembakkan ke fungsi isolasi memori lokal
                b_tp, b_tn, b_fp, b_fn = predict_and_evaluate_1c_batch(
                    embedding_model, buffer_waves, buffer_labels, kde_noise, kde_le
                )
                TP += b_tp; TN += b_tn; FP += b_fp; FN += b_fn
                
                buffer_waves.clear()
                buffer_labels.clear()
                
            # SHIELD MEMORY ANTI-LEAK 2: Pengosongan sesi backend berkala
            if (idx + 1) % 10000 == 0:
                print(f"\n[LOG 1C INDO INTERIM {idx+1}] TP:{TP} \u007c TN:{TN} \u007c FP:{FP} \u007c FN:{FN}")
                gc.collect() 
                K.clear_session()
                
        except Exception:
            FILE_KORUP += 1
            continue

    # Eksekusi sisa data pada kloter terakhir buffer
    if len(buffer_waves) > 0:
        b_tp, b_tn, b_fp, b_fn = predict_and_evaluate_1c_batch(
            embedding_model, buffer_waves, buffer_labels, kde_noise, kde_le
        )
        TP += b_tp; TN += b_tn; FP += b_fp; FN += b_fn

    # ==========================================================================
    # FASE 3: KALKULASI AKHIR METRIK REPLIKASI MURNI 1C DATA INDONESIA
    # ==========================================================================
    total_data = TP + TN + FP + FN
    akurasi = (TP + TN) / total_data if total_data > 0 else 0
    recall_tpr = TP / (TP + FN) if (TP + FN) > 0 else 0
    spesifisitas_tnr = TN / (TN + FP) if (TN + FP) > 0 else 0
    presisi_ppv = TP / (TP + FP) if (TP + FP) > 0 else 0
    f1_score = 2 * (presisi_ppv * recall_tpr) / (presisi_ppv + recall_tpr) if (presisi_ppv + recall_tpr) > 0 else 0

    print("\n=======================================================")
    print(" HASIL REPLIKASI MURNI 1C GAYA ZHI GENG (DATA INDONESIA)")
    print("=======================================================")
    print(f"Total Data Terproses  : {total_data:,} sampel gelombang")
    print(f"Total File Korup      : {FILE_KORUP:,} file (NaN/Inf atau Shape salah)")
    print(f"True Positives (TP)   : {TP:,}")
    print(f"True Negatives (TN)   : {TN:,}")
    print(f"False Positives (FP)  : {FP:,}")
    print(f"False Negatives (FN)  : {FN:,}")
    print("-------------------------------------------------------")
    print(f"Akurasi Global        : {akurasi:.4f} ({(akurasi*100):.2f}%)")
    print(f"Recall (TPR)          : {recall_tpr:.4f} ({(recall_tpr*100):.2f}%)")
    print(f"Spesifisitas (TNR)    : {spesifisitas_tnr:.4f} ({(spesifisitas_tnr*100):.2f}%)")
    print(f"Presisi (PPV)         : {presisi_ppv:.4f} ({(presisi_ppv*100):.2f}%)")
    print(f"F1-Score              : {f1_score:.4f} ({(f1_score*100):.2f}%)")
    print("=======================================================")

[INFO] Memuat Model dan Membangun Kurva PDF 1D Murni (Z-Axis)...
[INFO] Total Data Uji Indonesia Terdeteksi: 52,427 Gempa | 52,425 Noise

[INFO] Memulai eksekusi 1C murni Indonesia untuk 104,852 sampel...


Indonesia Pure 1C 100K:   8%|▊         | 8689/104852 [00:03<00:43, 2217.50it/s]


KeyboardInterrupt: 